In [1]:
# 데이터 불러오기
import pandas as pd

file_path = "/content/drive/MyDrive/JeonseGuard/실거래가/매매/오피스텔/202504_오피스텔_매매_실거래가.csv" # CSV 파일 경로 지정
df = pd.read_csv(file_path, encoding='cp949') # CP949 인코딩

In [2]:
# 상위 5개 확인
df.head()

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지
0,1,경기도 수원시 영통구 원천동,296-6,296,6,수원영통 아크로파크,90.9115,202504,30,"29,000",15,개인,개인,2006.0,삼성로320번길 35,-,중개거래,경기 수원시 영통구
1,2,제주특별자치도 제주시 노형동,722-2,722,2,메르헨하우스3,20.1400,202504,30,"4,050",1,개인,개인,2004.0,연북로 41,-,중개거래,제주 제주시
2,3,서울특별시 강서구 화곡동,936-1,936,1,골든팰리스,21.4700,202504,30,"28,800",11,개인,개인,2024.0,곰달래로16길 22,-,직거래,-
3,4,제주특별자치도 제주시 노형동,722-3,722,3,메르헨하우스2,20.1400,202504,30,"4,100",3,개인,개인,2004.0,연북로 45,-,중개거래,제주 제주시
4,5,서울특별시 중구 흥인동,142-1,142,1,위더스하임,29.7600,202504,30,"27,100",5,개인,개인,2018.0,퇴계로73길 20,-,중개거래,서울 중구


In [3]:
# 컬럼명 확인
print(df.columns)

Index(['NO', '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일',
       '거래금액(만원)', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형',
       '중개사소재지'],
      dtype='object')


In [4]:
# 특정 컬럼 값만 확인
df["거래금액(만원)"].head()

,거래금액(만원)
0,"29,000"
1,"4,050"
2,"28,800"
3,"4,100"
4,"27,100"


In [5]:
# 주택유형 컬럼 추가
df["주택유형"] = "오피스텔"

In [6]:
# 변경 매핑 딕셔너리 정의
renamed_columns = {
    "시군구": "address",
    "본번": "bun",
    "부번": "ji",
    "층": "floor",
    "전용면적(㎡)": "area",
    "계약년월": "contract_year_month",
    "거래금액(만원)": "price",
    "주택유형": "housing_type"
}

In [7]:
# 매핑에 해당하는 컬럼만 필터링
df = df.rename(columns=renamed_columns)
sale_df = df[list(renamed_columns.values())].copy()
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,경기도 수원시 영통구 원천동,296,6,15,90.9115,202504,"29,000",오피스텔
1,제주특별자치도 제주시 노형동,722,2,1,20.1400,202504,"4,050",오피스텔
2,서울특별시 강서구 화곡동,936,1,11,21.4700,202504,"28,800",오피스텔
3,제주특별자치도 제주시 노형동,722,3,3,20.1400,202504,"4,100",오피스텔
4,서울특별시 중구 흥인동,142,1,5,29.7600,202504,"27,100",오피스텔


In [8]:
# 쉼표 제거 및 문자열을 정수로 변환
sale_df["price"] = (
    sale_df["price"]
    .astype(str) # 문자열로 변환 (안전)
    .str.replace(",", "") # 쉼표 제거
    .astype(int) # 정수형으로 변환
    * 10000 # 만원 → 원 변환
)

In [9]:
# 변환된 price 컬럼 확인
sale_df[["price"]].head()

,price
0,290000000
1,40500000
2,288000000
3,41000000
4,271000000


In [10]:
# price 컬럼의 값을 쉼표가 포함된 문자열로 변환
sale_df["price"] = sale_df["price"].apply(lambda x: format(x, ","))
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,경기도 수원시 영통구 원천동,296,6,15,90.9115,202504,"290,000,000",오피스텔
1,제주특별자치도 제주시 노형동,722,2,1,20.1400,202504,"40,500,000",오피스텔
2,서울특별시 강서구 화곡동,936,1,11,21.4700,202504,"288,000,000",오피스텔
3,제주특별자치도 제주시 노형동,722,3,3,20.1400,202504,"41,000,000",오피스텔
4,서울특별시 중구 흥인동,142,1,5,29.7600,202504,"271,000,000",오피스텔


In [11]:
# 결측치 확인
missing_counts = sale_df.isnull().sum()
print("📌 결측치가 있는 컬럼: ", missing_counts[missing_counts > 0])

📌 결측치가 있는 컬럼:  Series([], dtype: int64)


In [12]:
# 전체 중복된 행의 수 확인
duplicate_count = sale_df.duplicated().sum()
print(f"📌 중복된 행의 수: {duplicate_count}")

📌 중복된 행의 수: 348


In [13]:
# 중복 제거 (기본: 모든 열 기준, keep='first')
sale_df = sale_df.drop_duplicates().reset_index(drop=True)

In [14]:
# INSERT 구문 생성
insert_header = """INSERT INTO transaction_sale_officetel (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES """

In [15]:
# 행별로 SQL 값 문자열 생성
values = []

In [16]:
# 각 행을 values 리스트에 추가
for _, row in sale_df.iterrows():
    values.append(f"('{row['address']}', '{row['bun']}', '{row['ji']}', '{row['floor']}', '{row['area']}', '{row['contract_year_month']}', '{row['price']}', '{row['housing_type']}', NOW(), NOW())")

In [17]:
# INSERT 구문 상위 5개만 출력
preview_sql = insert_header + ",\n       ".join(values[:5]) + ";"
print(preview_sql)

INSERT INTO transaction_sale_officetel (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES ('경기도 수원시 영통구 원천동', '296', '6', '15', '90.9115', '202504', '290,000,000', '오피스텔', NOW(), NOW()),
       ('제주특별자치도 제주시 노형동', '722', '2', '1', '20.14', '202504', '40,500,000', '오피스텔', NOW(), NOW()),
       ('서울특별시 강서구 화곡동', '936', '1', '11', '21.47', '202504', '288,000,000', '오피스텔', NOW(), NOW()),
       ('제주특별자치도 제주시 노형동', '722', '3', '3', '20.14', '202504', '41,000,000', '오피스텔', NOW(), NOW()),
       ('서울특별시 중구 흥인동', '142', '1', '5', '29.76', '202504', '271,000,000', '오피스텔', NOW(), NOW());


In [18]:
# INSERT 구문 조립
insert_sql = insert_header + ",\n       ".join(values) + ";"

In [19]:
# 파일 저장
file_name = "V40__insert_transaction_sale_officetel_202504.sql"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(insert_sql)

print(f"{file_name} 파일이 생성되었습니다.")

V40__insert_transaction_sale_officetel_202504.sql 파일이 생성되었습니다.
